
# Packages import

In [1]:
import os
import yaml
import requests
import pandas as pd
from datetime import datetime
import uuid
from requests.auth import HTTPBasicAuth
from bs4 import BeautifulSoup

ModuleNotFoundError: No module named 'yaml'

# Apollo scraper

In [ ]:

with open("config.yaml", "r", encoding="UTF-8") as yf:
    config = yaml.safe_load(yf)
username = config['credentials']['username']
password = config['credentials']['password']
credentials = HTTPBasicAuth(username, password)

In [ ]:

group_name = input("Enter group ID: ")
url = "https://planzajec.uek.krakow.pl/index.php?typ=G&id=252681&okres=2"
response = requests.get(url, auth=credentials)
response.encoding = "UTF-8"
print(response.status_code)

In [ ]:

print(response.text.encode().decode())

In [ ]:

page_dom = BeautifulSoup(response.text, "html.parser")

In [ ]:

group = page_dom.select_one("div.grupa").get_text(strip=True)
print(group)

In [ ]:

classes_tag = page_dom.select_one("table")
with open("temp.html", "w", encoding="UTF-8") as hf:
    hf.write(classes_tag.prettify())
classes = pd.read_html("temp.html", encoding="UTF-8")[0]
os.remove("temp.html")

In [ ]:
classes = classes.loc[
    classes['Typ'].isin(["ćwiczenia", "wykład", "egzamin"])
]

In [ ]:
classes[["Day", "Start time", "hyphen", "End time", "Duration"]] = classes["Dzień, godzina"].str.split(" ", expand=True)

In [ ]:

classes["Duration"] = classes["Duration"].map(lambda x: x.split('(')[1].split("g")[0])

In [ ]:

classes = classes.drop(['Dzień, godzina', 'hyphen'], axis=1)

In [ ]:

classes['Sala'] = classes['Sala'].str.replace(
    r"(lab\.).*",
    r"\1",
    regex=True
)

In [ ]:
if not os.path.exists("schedules"):
    os.mkdir("schedules")

In [ ]:

classes.to_csv(f"schedules/{group}.csv")

In [ ]:

classes

In [ ]:
def escape_ics(text):
    if text is None:
        return ""
    return str(text).replace("\\", "\\\\").replace(",", "\\,").replace(";", "\\;").replace("\n", "\\n")

ics = [
    "BEGIN:VCALENDAR",
    "VERSION:2.0",
    "PRODID:-//WebScraper//Plan zajec//PL",
    "CALSCALE:GREGORIAN"
]

for _, row in classes.iterrows():
    start = datetime.strptime(
        f"{row['Termin']} {row['Start time']}",
        "%Y-%m-%d %H:%M"
    )
    end = datetime.strptime(
        f"{row['Termin']} {row['End time']}",
        "%Y-%m-%d %H:%M"
    )

    title = f"{row['Przedmiot']} - {row['Typ']}"
    location = row["Sala"]
    description = f"Nauczyciel: {row['Nauczyciel']}"

    ics += [
        "BEGIN:VEVENT",
        f"UID:{uuid.uuid4()}",
        f"DTSTAMP:{datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')}",
        f"DTSTART;TZID=Europe/Warsaw:{start.strftime('%Y%m%dT%H%M%S')}",
        f"DTEND;TZID=Europe/Warsaw:{end.strftime('%Y%m%dT%H%M%S')}",
        f"SUMMARY:{escape_ics(title)}",
        f"LOCATION:{escape_ics(location)}",
        f"DESCRIPTION:{escape_ics(description)}",
        "END:VEVENT"
    ]

ics.append("END:VCALENDAR")

with open("plan_zajec.ics", "w", encoding="utf-8") as f:
    f.write("\n".join(ics))

print("Zapisano plik plan_zajec.ics")